# TheDecider — Evaluation & Promotion

Compares the current Production version of **TheDecider** (note classifier) against
the latest training run (or a specific run). Visual comparison is shown below;
use the Promote cell at the bottom to register a new Production version.


In [1]:
# ── Config ────────────────────────────────────────────────────────────────
MLFLOW_URI = "http://192.168.1.254:5000"
MODEL_NAME = "TheDecider"
RUN_ID     = None   # set to a specific run_id string to override 'use latest run'
N_VAL      = 2000   # validation samples to evaluate on


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import torch
import mlflow
from mlflow.tracking import MlflowClient
from torch.utils.data import DataLoader
from rustic_ml.training.setup import setup_mlflow, setup_device
from rustic_ml.the_decider.dataset import DeciderDataset
from rustic_ml.the_decider.inference import evaluate
from rustic_ml.mlflow_ui import show_register_widget, show_describe_widget

setup_mlflow(MLFLOW_URI, MODEL_NAME)
device = setup_device()
client = MlflowClient(MLFLOW_URI)
print(f"Device: {device}")

val_ds     = DeciderDataset(n_samples=N_VAL)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
print(f"Val samples: {len(val_ds)}")

In [ ]:
# ── Load production model ─────────────────────────────────────────────────
prod_model = None
prod_metrics = {}

prod_versions = client.get_latest_versions(MODEL_NAME, stages=["Production"])
if prod_versions:
    prod_model = mlflow.pytorch.load_model(prod_versions[0].source, map_location=device)
    prod_model.eval()
    print(f"Production: version {prod_versions[0].version} ({prod_versions[0].run_id[:8]})")
    prod_metrics = evaluate(prod_model, val_loader, device)
else:
    print("No Production version registered yet — candidate will be evaluated against nothing.")


In [ ]:
# ── Load candidate model ──────────────────────────────────────────────────
if RUN_ID is None:
    exp = client.get_experiment_by_name(MODEL_NAME)
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=["start_time DESC"],
        max_results=1,
    )
    RUN_ID = runs[0].info.run_id

cand_model = mlflow.pytorch.load_model(f"runs:/{RUN_ID}/model", map_location=device)
cand_model.eval()
print(f"Candidate run: {RUN_ID}")
cand_metrics = evaluate(cand_model, val_loader, device)


In [ ]:
# ── Metrics table ─────────────────────────────────────────────────────────
import pandas as pd

rows = []
for key in set(list(prod_metrics) + list(cand_metrics)):
    rows.append({
        "metric":    key,
        "production": prod_metrics.get(key, "—"),
        "candidate":  cand_metrics.get(key, "—"),
    })
display(pd.DataFrame(rows).set_index("metric"))


In [ ]:
# ── Visual comparison ─────────────────────────────────────────────────────
# TODO: implement once TheDecider and its dataset are ready.
# Suggested plots:
#   - Per-note top-1 accuracy bar chart (production vs candidate overlay)
#   - Confusion matrix on note quartiles
#   - Top-5 accuracy by note range
print("Visual comparison: TODO")


In [ ]:
# ── Promote ───────────────────────────────────────────────────────────────
# Register the candidate as the new Production version.
# Only do this if the candidate beats production on top1_accuracy.
prod_top1 = prod_metrics.get("top1_accuracy", 0.0)
cand_top1 = cand_metrics.get("top1_accuracy", 0.0)
if cand_top1 > prod_top1:
    print(f"Candidate improves top-1 accuracy: {prod_top1:.3f} → {cand_top1:.3f}")
else:
    print(f"Candidate does NOT improve top-1 accuracy ({cand_top1:.3f} vs {prod_top1:.3f}). Promote anyway?")

show_register_widget(RUN_ID, "model", model_name=MODEL_NAME)
show_describe_widget(RUN_ID, MLFLOW_URI)
